# Node classification graph-layout visualization for IVN graph windows

Notebook này chỉ giữ **Phần 2b** ở dạng graph layout:

1. Load graph metadata, checkpoint, và mô hình multitask.
2. Tự động chọn một **correct mixed graph**.
3. Tự động chọn một **imperfect mixed graph** có một số attack nodes được dự đoán đúng và ít nhất một node bị sai.
4. Vẽ đúng **hai graph-layout figures**:
   - correct case,
   - imperfect case.

Notebook này không chạy relation heatmap, node occlusion, KAN head visualization, timeline, hoặc attack-probability plot.


In [ ]:
from pathlib import Path
import sys

# ============================================================
# CONFIGURATION
# ============================================================
PROJECT_ROOT = Path('..').resolve()

GRAPH_FOLDER = PROJECT_ROOT / 'data/2017-subaru-forester/graphs_subsample_node_classification_v2'
CKPT_PATH = PROJECT_ROOT / 'save/graph_attention_ffn_kan_multitask_turning/graph_attention_ffn_kan_multitask_turning_best_joint.pth'

# Use 'cuda:0', 'cuda:1', or 'cpu'
DEVICE = 'cpu'

# Data split for visualization
SPLIT = 'test'

# Label settings
NORMAL_LABEL = 'normal'      # if not found, notebook will auto-resolve
ATTACK_LABEL = None          # None = select first non-normal label for initial loading
NORMAL_OCCURRENCE = 0
ATTACK_OCCURRENCE = 0

# Model class used for the trained checkpoint
MODEL_MODULE = 'networks.graph_attention_ffn_kan_multitask_updated'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('GRAPH_FOLDER =', GRAPH_FOLDER)
print('CKPT_PATH    =', CKPT_PATH)
print('DEVICE       =', DEVICE)
print('MODEL_MODULE =', MODEL_MODULE)


In [ ]:
import os
import json
import math
import importlib
import inspect
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap

plt.rcParams['figure.dpi'] = 120


## 1. Load metadata, sample graphs, checkpoint, and model


In [ ]:

def load_table(base: Path) -> pd.DataFrame:
    pq = base.with_suffix('.parquet')
    csv = base.with_suffix('.csv')
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    raise FileNotFoundError(f'Cannot find {pq} or {csv}')


def normalize_label_name(x) -> str:
    return str(x).strip().lower()


def infer_label_mapping(graph_folder: Path) -> Dict[int, str]:
    mapping = {}
    for split in ['train', 'val', 'test']:
        try:
            df = load_table(graph_folder / f'graph_index_{split}')
            if 'y' in df.columns and 'window_label' in df.columns:
                pairs = df[['y', 'window_label']].drop_duplicates()
                for _, row in pairs.iterrows():
                    mapping[int(row['y'])] = str(row['window_label'])
        except Exception:
            continue
    return dict(sorted(mapping.items(), key=lambda kv: kv[0]))


def graph_dict_to_data(graph: dict) -> Data:
    """
    Convert one saved graph dictionary to PyG Data.

    Important for this notebook:
    - keep graph-level fields for graph classification/explainability;
    - also keep node_y/node_mask/node_is_attack because the checkpoint was
      trained with multitask node supervision.
    """
    data = Data(
        x=graph['x'].float(),
        edge_index=graph['edge_index'].long(),
        edge_attr=graph['edge_attr'].float(),
        edge_type=graph['edge_type'].long(),
        id_token=graph['id_index'].long() if 'id_index' in graph else graph['id_token'].long(),
        y=graph['y'].view(-1).long(),
    )

    # Optional message/window metadata
    if 'arbitration_id' in graph:
        data.arbitration_id = graph['arbitration_id'].long()
    if 'timestamp' in graph:
        data.timestamp = graph['timestamp'].float()
    if 'msg_idx_in_file' in graph:
        data.msg_idx_in_file = graph['msg_idx_in_file'].long()

    # Node-level labels used by the multitask checkpoint
    if 'node_y' in graph:
        data.node_y = graph['node_y'].long()
    if 'node_mask' in graph:
        data.node_mask = graph['node_mask'].bool()
    else:
        data.node_mask = torch.ones(data.x.size(0), dtype=torch.bool)
    if 'node_is_attack' in graph:
        data.node_is_attack = graph['node_is_attack'].long()

    data.graph_id = graph.get('graph_id', 'unknown_graph')
    data.window_label = graph.get('window_label', None)
    data.source_class = graph.get('source_class', None)
    data.meta = graph.get('meta', {})
    return data




def clone_data(data: Data) -> Data:
    """
    Clone a PyG Data object.

    This helper is needed by the node-classification visualization code.
    It clones tensor attributes and keeps non-tensor metadata unchanged.
    """
    out = Data()
    for k in data.keys():
        v = getattr(data, k)
        if torch.is_tensor(v):
            out[k] = v.clone()
        else:
            out[k] = v
    return out


def get_graph_label_name(raw_graph: dict, label_mapping: Dict[int, str]) -> str:
    if 'window_label' in raw_graph:
        return str(raw_graph['window_label'])
    y = int(raw_graph['y'].view(-1)[0].item())
    return label_mapping.get(y, f'class_{y}')


def load_graph_metadata(graph_folder: Path) -> Dict[str, Any]:
    meta_path = graph_folder / 'graph_metadata.json'
    if meta_path.exists():
        with open(meta_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {
        'relation_to_index': {'temporal': 0, 'same_id': 1, 'payload_sim': 2, 'timing_aff': 3},
        'index_to_relation': {'0': 'temporal', '1': 'same_id', '2': 'payload_sim', '3': 'timing_aff'},
    }


def infer_num_ids_from_shards(graph_folder: Path) -> int:
    max_id = -1
    for split in ['train', 'val', 'test']:
        split_dir = graph_folder / split
        if not split_dir.exists():
            continue
        for shard_path in sorted(split_dir.glob(f'graphs_{split}_shard*.pt')):
            shard_graphs = torch.load(shard_path, map_location='cpu', weights_only=False)
            for g in shard_graphs:
                if 'id_index' in g:
                    max_id = max(max_id, int(g['id_index'].max().item()))
    if max_id < 0:
        raise RuntimeError('Cannot infer num_ids from shards.')
    return max_id + 1


def iter_raw_graphs(graph_folder: Path, split: str):
    split_dir = graph_folder / split
    shard_paths = sorted(split_dir.glob(f'graphs_{split}_shard*.pt'))
    for shard_path in shard_paths:
        shard_graphs = torch.load(shard_path, map_location='cpu', weights_only=False)
        for raw_g in shard_graphs:
            yield raw_g


def get_split_label_counts(graph_folder: Path, split: str, label_mapping: Dict[int, str]) -> pd.DataFrame:
    df = load_table(graph_folder / f'graph_index_{split}').copy()

    if 'window_label' in df.columns:
        out = (
            df['window_label']
            .astype(str)
            .value_counts()
            .rename_axis('label_name')
            .reset_index(name='count')
        )
    elif 'y' in df.columns:
        out = (
            df['y']
            .astype(int)
            .value_counts()
            .rename_axis('label_id')
            .reset_index(name='count')
        )
        out['label_name'] = out['label_id'].map(label_mapping).fillna(out['label_id'].astype(str))
        out = out[['label_id', 'label_name', 'count']]
    else:
        raise ValueError('graph_index file has neither window_label nor y.')
    return out


def auto_find_normal_label(label_names: List[str]) -> Optional[str]:
    candidates = [str(x) for x in label_names]
    norm_map = {normalize_label_name(x): str(x) for x in candidates}
    for key in ['normal', 'benign', 'non_attack', 'non-attack', 'class_0', '0']:
        if key in norm_map:
            return norm_map[key]
    return None


def select_graph_by_label(
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    target_label: str,
    occurrence: int = 0,
) -> Tuple[dict, Data, str]:
    target_norm = normalize_label_name(target_label)
    seen = 0
    for raw_g in iter_raw_graphs(graph_folder, split):
        label_name = get_graph_label_name(raw_g, label_mapping)
        if normalize_label_name(label_name) == target_norm:
            if seen == occurrence:
                return raw_g, graph_dict_to_data(raw_g), label_name
            seen += 1
    raise ValueError(f'Cannot find label={target_label} occurrence={occurrence} in split={split}')


def select_first_attack_graph(
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    normal_label: str,
    occurrence: int = 0,
) -> Tuple[dict, Data, str]:
    normal_norm = normalize_label_name(normal_label)
    seen = 0
    for raw_g in iter_raw_graphs(graph_folder, split):
        label_name = get_graph_label_name(raw_g, label_mapping)
        if normalize_label_name(label_name) != normal_norm:
            if seen == occurrence:
                return raw_g, graph_dict_to_data(raw_g), label_name
            seen += 1
    raise ValueError(f'Cannot find attack graph in split={split}')


def import_model_class(module_name: str):
    mod = importlib.import_module(module_name)
    if not hasattr(mod, 'GraphAttentionKAN'):
        raise ImportError(f'Module {module_name} has no GraphAttentionKAN')
    return mod.GraphAttentionKAN


def build_model_from_checkpoint(
    ckpt_path: Path,
    sample_data: Data,
    graph_folder: Path,
    device: str = 'cpu',
    model_module: str = 'networks.graph_attention_ffn_kan_multitask_updated',
    strict_load: bool = True,
):
    """
    Build GraphAttentionKAN exactly as it was trained, then load checkpoint.

    The previous notebook version did not pass num_node_classes and
    node_head_from_layer, so the local model was created without node_head.
    That produced:
        Unexpected keys: node_head.*

    This version reconstructs the multitask model from checkpoint args.
    """
    GraphAttentionKAN = import_model_class(model_module)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    args = ckpt.get('args', {})

    label_mapping = ckpt.get('label_mapping', infer_label_mapping(graph_folder))
    label_mapping = {int(k): str(v) for k, v in label_mapping.items()}
    label_mapping = dict(sorted(label_mapping.items(), key=lambda kv: kv[0]))
    num_graph_classes = len(label_mapping)

    # Node label mapping saved by the multitask trainer if available.
    # For node_target='node_y', num_node_classes should be the same as graph classes.
    node_label_mapping = ckpt.get('node_label_mapping', None)
    if isinstance(node_label_mapping, dict) and len(node_label_mapping) > 0:
        node_label_mapping = {int(k): str(v) for k, v in node_label_mapping.items()}
        num_node_classes = len(node_label_mapping)
    else:
        if bool(args.get('enable_node_task', True)):
            if args.get('node_target', 'node_y') == 'node_is_attack':
                num_node_classes = 2
                node_label_mapping = {0: 'normal', 1: 'attack'}
            else:
                num_node_classes = num_graph_classes
                node_label_mapping = label_mapping.copy()
        else:
            num_node_classes = None
            node_label_mapping = {}

    # Prefer checkpoint shape because it is the exact value used by the trained model.
    # The model defines nn.Embedding(num_ids + 1, id_emb_dim), so num_ids = rows - 1.
    state_dict = ckpt['model']
    if 'id_embedding.weight' in state_dict:
        num_ids = int(state_dict['id_embedding.weight'].shape[0]) - 1
    else:
        num_ids = infer_num_ids_from_shards(graph_folder)

    ctor_kwargs = dict(
        node_feat_dim=int(sample_data.x.size(1)),
        edge_attr_dim=int(sample_data.edge_attr.size(1)),
        num_classes=num_graph_classes,
        num_ids=num_ids,

        # Critical for multitask checkpoint loading
        num_node_classes=num_node_classes,
        node_head_from_layer=args.get('node_head_from_layer', -1),

        hidden_dim=args.get('hidden_dim', 128),
        num_layers=args.get('num_layers', 3),
        heads=args.get('heads', 2),
        id_emb_dim=args.get('id_emb_dim', 32),
        rel_emb_dim=args.get('rel_emb_dim', 8),
        num_relations=args.get('num_relations', 4),
        dropout=args.get('dropout', 0.2),
        ffn_ratio=args.get('ffn_ratio', 2.0),

        block_kan_grid_size=args.get('block_kan_grid_size', 3),
        block_kan_spline_order=args.get('block_kan_spline_order', 3),
        block_kan_scale_noise=args.get('block_kan_scale_noise', 0.1),
        block_kan_scale_base=args.get('block_kan_scale_base', 1.0),
        block_kan_scale_spline=args.get('block_kan_scale_spline', 1.0),

        kan_hidden=args.get('kan_hidden', 64),
        kan_grid_size=args.get('kan_grid_size', 3),
        kan_spline_order=args.get('kan_spline_order', 3),
        kan_scale_noise=args.get('kan_scale_noise', 0.1),
        kan_scale_base=args.get('kan_scale_base', 1.0),
        kan_scale_spline=args.get('kan_scale_spline', 1.0),
    )

    # Keep compatibility if local class has slightly different signature.
    sig = inspect.signature(GraphAttentionKAN.__init__)
    ctor_kwargs = {k: v for k, v in ctor_kwargs.items() if k in sig.parameters}

    model = GraphAttentionKAN(**ctor_kwargs)

    if strict_load:
        model.load_state_dict(state_dict, strict=True)
        missing, unexpected = [], []
    else:
        missing, unexpected = model.load_state_dict(state_dict, strict=False)

    model = model.to(device).eval()

    print('Loaded checkpoint:', ckpt_path)
    print('Missing keys   :', missing)
    print('Unexpected keys:', unexpected)
    print('num_classes    :', num_graph_classes)
    print('num_node_classes:', num_node_classes)
    print('node_target    :', args.get('node_target', 'node_y'))
    print('node_head_from_layer:', args.get('node_head_from_layer', -1))
    print('num_ids        :', num_ids)

    if len(missing) == 0 and len(unexpected) == 0:
        print('✅ Checkpoint loaded successfully. Model architecture matches the multitask checkpoint.')
    else:
        print('⚠️  CẢNH BÁO: model class local và checkpoint chưa khớp hoàn toàn.')

    return model, ckpt, label_mapping


def forward_logits(model, data: Data, device: str = 'cpu'):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        logits = model(data)
    return logits.cpu()


def get_graph_embedding(model, data: Data, device: str = 'cpu') -> torch.Tensor:
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        try:
            out = model(data, return_graph_embedding=True)
            if isinstance(out, tuple) and len(out) == 2:
                _, g = out
                return g.detach().cpu()
        except TypeError:
            pass
        h = model.encode_nodes(data)
        batch = getattr(data, 'batch', None)
        if batch is None:
            batch = torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        g = model.readout(h, batch)
        return g.detach().cpu()


def get_logits_from_embedding(model, g: torch.Tensor, device: str = 'cpu') -> torch.Tensor:
    model.eval()
    g = g.to(device)
    with torch.no_grad():
        logits = model.head(g)
    return logits.detach().cpu()


In [ ]:

graph_meta = load_graph_metadata(GRAPH_FOLDER)
label_mapping = infer_label_mapping(GRAPH_FOLDER)
label_counts = get_split_label_counts(GRAPH_FOLDER, SPLIT, label_mapping)

display(label_counts)
print('label_mapping =', label_mapping)
print('relation mapping =', graph_meta.get('index_to_relation', {}))


In [ ]:

available_labels = label_counts['label_name'].astype(str).tolist()

resolved_normal_label = NORMAL_LABEL
if NORMAL_LABEL is None or normalize_label_name(NORMAL_LABEL) not in [normalize_label_name(x) for x in available_labels]:
    auto_label = auto_find_normal_label(available_labels)
    if auto_label is None:
        raise ValueError(f'Cannot resolve NORMAL_LABEL automatically. Available labels: {available_labels}')
    resolved_normal_label = auto_label

print('Resolved NORMAL_LABEL =', resolved_normal_label)

raw_normal, data_normal, normal_label_name = select_graph_by_label(
    GRAPH_FOLDER, SPLIT, label_mapping, resolved_normal_label, occurrence=NORMAL_OCCURRENCE
)

if ATTACK_LABEL is None:
    attack_candidates = [x for x in available_labels if normalize_label_name(x) != normalize_label_name(resolved_normal_label)]
    if len(attack_candidates) == 0:
        raise ValueError('No attack label found in this split.')
    resolved_attack_label = attack_candidates[ATTACK_OCCURRENCE]
else:
    resolved_attack_label = ATTACK_LABEL

raw_attack, data_attack, attack_label_name = select_graph_by_label(
    GRAPH_FOLDER, SPLIT, label_mapping, resolved_attack_label, occurrence=0
)

print('Normal graph:', data_normal.graph_id, '| label =', normal_label_name, '| nodes =', data_normal.x.size(0))
print('Attack graph:', data_attack.graph_id, '| label =', attack_label_name, '| nodes =', data_attack.x.size(0))

model, ckpt, ckpt_label_mapping = build_model_from_checkpoint(
    CKPT_PATH, data_normal, GRAPH_FOLDER, device=DEVICE, model_module=MODEL_MODULE
)


## 2. Node classification graph-layout visualization

Phần này dùng trực tiếp node head của checkpoint multitask.

Mục tiêu là tạo hai ví dụ định tính cho section **Node Classification Performance**:

- **Correct case**: mixed graph/window mà các attack nodes được định vị đúng.
- **Imperfect case**: mixed graph/window trong đó mô hình bắt đúng một số attack nodes nhưng vẫn có ít nhất một node bị sai.

Mỗi case được vẽ bằng graph layout với hai panel:
- panel trái: ground-truth node status;
- panel phải: predicted node status.


In [ ]:

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os


def parse_multitask_output(output):
    """Return graph_logits and node_logits from GraphAttentionKAN output."""
    if isinstance(output, dict):
        return output.get('graph_logits', None), output.get('node_logits', None)
    if isinstance(output, (tuple, list)):
        if len(output) >= 2:
            return output[0], output[1]
        if len(output) == 1:
            return output[0], None
    return output, None


def resolve_normal_class_id(label_mapping: Dict[int, str], normal_label: str = 'normal') -> int:
    normal_norm = normalize_label_name(normal_label)
    for idx, name in label_mapping.items():
        if normalize_label_name(name) == normal_norm:
            return int(idx)
    # fallback: class 0 is usually normal in this pipeline
    return 0


def get_node_label_names(label_ids, label_mapping: Dict[int, str]):
    return [label_mapping.get(int(i), f'class_{int(i)}') for i in label_ids]


def get_single_graph_node_classification(
    model,
    data: Data,
    label_mapping: Dict[int, str],
    device: str = 'cpu',
    node_target: str = 'node_y',
    normal_label: str = 'normal',
):
    """
    Run one graph/window through the model and collect node-level predictions.

    Returns a dictionary containing multiclass node prediction and binary
    normal/attack conversion for visualization.
    """
    model.eval()
    normal_class_id = resolve_normal_class_id(label_mapping, normal_label)

    data_in = clone_data(data).to(device)
    data_in.x = data_in.x.float()
    data_in.edge_index = data_in.edge_index.long()
    if hasattr(data_in, 'edge_attr'):
        data_in.edge_attr = data_in.edge_attr.float()
    if hasattr(data_in, 'edge_type'):
        data_in.edge_type = data_in.edge_type.long()
    if hasattr(data_in, 'id_token'):
        data_in.id_token = data_in.id_token.long()

    with torch.no_grad():
        output = model(data_in, return_node_logits=True)
        graph_logits, node_logits = parse_multitask_output(output)

        if graph_logits is None:
            raise RuntimeError('Model did not return graph_logits.')
        if node_logits is None:
            raise RuntimeError('Model did not return node_logits. Check num_node_classes and checkpoint loading.')

        graph_prob = F.softmax(graph_logits, dim=1)
        graph_pred = graph_prob.argmax(dim=1)

        node_prob = F.softmax(node_logits, dim=1)
        node_pred = node_prob.argmax(dim=1)
        node_conf = node_prob.max(dim=1).values

        # Attack probability:
        # for node_y multiclass: P(attack) = 1 - P(normal)
        # for node_is_attack binary: P(attack) = P(class 1)
        if node_prob.size(1) == 2 and node_target == 'node_is_attack':
            attack_prob = node_prob[:, 1]
            pred_is_attack = node_pred == 1
        else:
            attack_prob = 1.0 - node_prob[:, normal_class_id]
            pred_is_attack = node_pred != normal_class_id

    if not hasattr(data, node_target):
        raise AttributeError(f'Data object does not have node target: {node_target}')

    node_true = getattr(data, node_target).view(-1).detach().cpu().long()
    node_pred_cpu = node_pred.detach().cpu().long()
    node_prob_cpu = node_prob.detach().cpu()
    node_conf_cpu = node_conf.detach().cpu()
    attack_prob_cpu = attack_prob.detach().cpu()

    if hasattr(data, 'node_mask'):
        node_mask = data.node_mask.view(-1).detach().cpu().bool()
    else:
        node_mask = torch.ones_like(node_true, dtype=torch.bool)

    if node_target == 'node_is_attack':
        true_is_attack = node_true == 1
    else:
        true_is_attack = node_true != normal_class_id

    # Basic node-level summary on the selected graph
    valid = node_mask.numpy().astype(bool)
    true_bin = true_is_attack.numpy()[valid].astype(int)
    pred_bin = pred_is_attack.detach().cpu().numpy()[valid].astype(int)
    bin_acc = float((true_bin == pred_bin).mean()) if valid.sum() > 0 else 0.0

    return {
        'graph_id': getattr(data, 'graph_id', 'unknown_graph'),
        'window_label': getattr(data, 'window_label', None),
        'source_class': getattr(data, 'source_class', None),
        'graph_true_idx': int(data.y.view(-1)[0].item()),
        'graph_pred_idx': int(graph_pred.item()),
        'graph_true_name': label_mapping.get(int(data.y.view(-1)[0].item()), f'class_{int(data.y.view(-1)[0].item())}'),
        'graph_pred_name': label_mapping.get(int(graph_pred.item()), f'class_{int(graph_pred.item())}'),
        'graph_prob': graph_prob.detach().cpu(),
        'node_true': node_true,
        'node_pred': node_pred_cpu,
        'node_prob': node_prob_cpu,
        'node_conf': node_conf_cpu,
        'attack_prob': attack_prob_cpu,
        'node_mask': node_mask,
        'true_is_attack': true_is_attack.detach().cpu().bool(),
        'pred_is_attack': pred_is_attack.detach().cpu().bool(),
        'normal_class_id': normal_class_id,
        'normal_label': label_mapping.get(normal_class_id, str(normal_label)),
        'binary_attack_acc': bin_acc,
        'num_attack_true': int(true_is_attack[node_mask].sum().item()),
        'num_attack_pred': int(pred_is_attack.detach().cpu()[node_mask].sum().item()),
        'num_nodes': int(node_mask.sum().item()),
    }


def node_classification_result_to_df(result, data: Data, label_mapping: Dict[int, str]) -> pd.DataFrame:
    """Convert node classification result into an easy-to-check dataframe."""
    n = int(result['node_true'].numel())
    df = pd.DataFrame({
        'node_idx': np.arange(n),
        'node_mask': result['node_mask'].numpy().astype(bool),
        'true_label_id': result['node_true'].numpy().astype(int),
        'pred_label_id': result['node_pred'].numpy().astype(int),
        'true_label': get_node_label_names(result['node_true'].numpy(), label_mapping),
        'pred_label': get_node_label_names(result['node_pred'].numpy(), label_mapping),
        'true_is_attack': result['true_is_attack'].numpy().astype(bool),
        'pred_is_attack': result['pred_is_attack'].numpy().astype(bool),
        'attack_prob': result['attack_prob'].numpy(),
        'node_confidence': result['node_conf'].numpy(),
    })
    if hasattr(data, 'arbitration_id'):
        df['arbitration_id'] = data.arbitration_id.detach().cpu().numpy().astype(int)
    if hasattr(data, 'timestamp'):
        df['timestamp'] = data.timestamp.detach().cpu().numpy()
    if hasattr(data, 'msg_idx_in_file'):
        df['msg_idx_in_file'] = data.msg_idx_in_file.detach().cpu().numpy().astype(int)
    df['binary_correct'] = df['true_is_attack'] == df['pred_is_attack']
    df['multiclass_correct'] = df['true_label_id'] == df['pred_label_id']
    return df


def find_mixed_node_graph(
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    normal_label: str = 'normal',
    prefer_label: Optional[str] = None,
    max_scan: int = 5000,
):
    """
    Find a graph/window containing both normal and attack nodes.

    If no mixed graph is found within max_scan, return None.
    """
    normal_id = resolve_normal_class_id(label_mapping, normal_label)
    prefer_norm = normalize_label_name(prefer_label) if prefer_label is not None else None
    scanned = 0

    # First pass: prefer selected attack label if provided
    for raw_g in iter_raw_graphs(graph_folder, split):
        scanned += 1
        if scanned > max_scan:
            break
        label_name = get_graph_label_name(raw_g, label_mapping)
        if prefer_norm is not None and normalize_label_name(label_name) != prefer_norm:
            continue
        data = graph_dict_to_data(raw_g)
        if not hasattr(data, 'node_y'):
            continue
        y = data.node_y.view(-1)
        has_normal = bool((y == normal_id).any().item())
        has_attack = bool((y != normal_id).any().item())
        if has_normal and has_attack:
            return raw_g, data, label_name

    # Second pass: any mixed graph
    scanned = 0
    for raw_g in iter_raw_graphs(graph_folder, split):
        scanned += 1
        if scanned > max_scan:
            break
        data = graph_dict_to_data(raw_g)
        if not hasattr(data, 'node_y'):
            continue
        label_name = get_graph_label_name(raw_g, label_mapping)
        y = data.node_y.view(-1)
        has_normal = bool((y == normal_id).any().item())
        has_attack = bool((y != normal_id).any().item())
        if has_normal and has_attack:
            return raw_g, data, label_name

    return None



def _binary_node_metrics_from_result(result: Dict[str, Any]) -> Dict[str, float]:
    """Compute binary normal/attack node metrics for one graph result."""
    mask = result['node_mask'].numpy().astype(bool)
    true_bin = result['true_is_attack'].numpy()[mask].astype(int)
    pred_bin = result['pred_is_attack'].numpy()[mask].astype(int)

    tp = int(((true_bin == 1) & (pred_bin == 1)).sum())
    tn = int(((true_bin == 0) & (pred_bin == 0)).sum())
    fp = int(((true_bin == 0) & (pred_bin == 1)).sum())
    fn = int(((true_bin == 1) & (pred_bin == 0)).sum())

    total = max(1, len(true_bin))
    acc = (tp + tn) / total
    attack_recall = tp / max(1, tp + fn)
    attack_precision = tp / max(1, tp + fp)
    attack_f1 = 2 * attack_precision * attack_recall / max(1e-12, attack_precision + attack_recall)
    exact_binary_match = bool(np.array_equal(true_bin, pred_bin))

    return {
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'num_nodes': int(total),
        'num_true_attack': int(tp + fn),
        'num_pred_attack': int(tp + fp),
        'binary_acc': float(acc),
        'attack_recall': float(attack_recall),
        'attack_precision': float(attack_precision),
        'attack_f1': float(attack_f1),
        'exact_binary_match': exact_binary_match,
        'graph_correct': bool(result['graph_true_idx'] == result['graph_pred_idx']),
    }



def summarize_node_classification(result: Dict[str, Any], name: str = 'selected graph') -> Dict[str, Any]:
    """
    Print a compact summary for a selected node-classification case.

    This function is used after selecting successful/imperfect graph windows.
    It is intentionally lightweight and only depends on the result dictionary
    returned by get_single_graph_node_classification() or the fast selector.
    """
    metrics = _binary_node_metrics_from_result(result)

    node_mask = result['node_mask'].detach().cpu().numpy().astype(bool)
    node_true = result['node_true'].detach().cpu().numpy().astype(int)
    node_pred = result['node_pred'].detach().cpu().numpy().astype(int)

    if node_mask.sum() > 0:
        multiclass_acc = float((node_true[node_mask] == node_pred[node_mask]).mean())
    else:
        multiclass_acc = 0.0

    metrics['multiclass_acc'] = multiclass_acc

    print('=' * 100)
    print(f'Node-classification summary: {name}')
    print('  graph_id           :', result.get('graph_id', 'unknown_graph'))
    print('  graph true/pred    :', result.get('graph_true_name', 'unknown'), '->', result.get('graph_pred_name', 'unknown'))
    print('  graph_correct      :', metrics['graph_correct'])
    print('  true attack nodes  :', metrics['num_true_attack'])
    print('  pred attack nodes  :', metrics['num_pred_attack'])
    print('  TP / FP / FN / TN  :', metrics['tp'], '/', metrics['fp'], '/', metrics['fn'], '/', metrics['tn'])
    print('  binary node acc    :', f"{metrics['binary_acc']:.4f}")
    print('  attack recall      :', f"{metrics['attack_recall']:.4f}")
    print('  attack precision   :', f"{metrics['attack_precision']:.4f}")
    print('  attack F1          :', f"{metrics['attack_f1']:.4f}")
    print('  multiclass acc     :', f"{metrics['multiclass_acc']:.4f}")
    print('=' * 100)

    return metrics


def find_successful_node_classification_graph(
    model,
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    device: str = 'cpu',
    node_target: str = 'node_y',
    normal_label: str = 'normal',
    prefer_label: Optional[str] = None,
    max_scan: int = 20000,
    min_attack_nodes: int = 1,
    require_graph_correct: bool = True,
    require_exact_binary_match: bool = True,
):
    """
    Find an informative mixed graph where node classification is successful.

    Priority:
      1) graph contains both normal and attack nodes;
      2) graph-level prediction is correct, if require_graph_correct=True;
      3) binary node status is exactly correct, if require_exact_binary_match=True;
      4) prefer the requested attack label if prefer_label is given.

    If no perfect case is found, return the best available mixed graph according to
    attack recall, binary accuracy, attack precision, and graph correctness.
    """
    normal_id = resolve_normal_class_id(label_mapping, normal_label)
    prefer_norm = normalize_label_name(prefer_label) if prefer_label is not None else None

    def candidate_iterator(prefer_only: bool):
        scanned = 0
        for raw_g in iter_raw_graphs(graph_folder, split):
            scanned += 1
            if scanned > max_scan:
                break
            label_name = get_graph_label_name(raw_g, label_mapping)
            if prefer_only and prefer_norm is not None and normalize_label_name(label_name) != prefer_norm:
                continue
            data = graph_dict_to_data(raw_g)
            if not hasattr(data, node_target):
                continue
            node_y = getattr(data, node_target).view(-1)
            if node_target == 'node_is_attack':
                true_attack = node_y == 1
                true_normal = node_y == 0
            else:
                true_attack = node_y != normal_id
                true_normal = node_y == normal_id
            num_attack = int(true_attack.sum().item())
            has_normal = bool(true_normal.any().item())
            has_attack = num_attack >= min_attack_nodes
            if not (has_normal and has_attack):
                continue
            yield raw_g, data, label_name

    best = None
    best_score = None

    # pass 1: preferred label if possible; pass 2: any mixed label
    for prefer_only in ([True, False] if prefer_norm is not None else [False]):
        for raw_g, data, label_name in candidate_iterator(prefer_only=prefer_only):
            result = get_single_graph_node_classification(
                model=model,
                data=data,
                label_mapping=label_mapping,
                device=device,
                node_target=node_target,
                normal_label=normal_label,
            )
            metrics = _binary_node_metrics_from_result(result)

            graph_ok = (metrics['graph_correct'] or not require_graph_correct)
            node_ok = (metrics['exact_binary_match'] or not require_exact_binary_match)
            if graph_ok and node_ok:
                return {
                    'raw_graph': raw_g,
                    'data': data,
                    'label_name': label_name,
                    'result': result,
                    'metrics': metrics,
                    'selection_status': 'perfect_match',
                }

            # Keep best fallback. Higher score is better.
            score = (
                int(metrics['graph_correct']),
                metrics['attack_recall'],
                metrics['binary_acc'],
                metrics['attack_precision'],
                metrics['attack_f1'],
                -abs(metrics['num_true_attack'] - metrics['num_pred_attack']),
            )
            if best_score is None or score > best_score:
                best_score = score
                best = {
                    'raw_graph': raw_g,
                    'data': data,
                    'label_name': label_name,
                    'result': result,
                    'metrics': metrics,
                    'selection_status': 'best_fallback',
                }

        # If pass 1 already found a good fallback with preferred label, do not stop;
        # pass 2 may still find a perfect graph from another class.

    return best



def find_successful_node_classification_graph_strict(
    model,
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    device: str = 'cpu',
    node_target: str = 'node_y',
    normal_label: str = 'normal',
    prefer_label: Optional[str] = None,
    max_scan: int = 30000,
    min_true_attack_nodes: int = 3,
    max_true_attack_nodes: Optional[int] = 10,
    min_tp_attack_nodes: int = 2,
    min_binary_node_acc: float = 0.99,
    max_attack_ratio: float = 0.20,
    require_graph_correct: bool = True,
):
    """
    Select an informative attack/mixed graph for Figure 2b.

    Strict target criteria:
      - the graph contains both normal nodes and attack nodes;
      - number of true attack nodes >= min_true_attack_nodes;
      - number of correctly detected attack nodes TP >= min_tp_attack_nodes;
      - graph-level prediction is correct, if require_graph_correct=True;
      - node-level attack recall = 1.0;
      - node-level attack precision = 1.0;
      - binary node accuracy is close to 1.0, controlled by min_binary_node_acc.

    If no strict-perfect graph is found, the function returns the best fallback
    that still satisfies the minimum number of true attack nodes. The fallback is
    ranked by graph correctness, attack recall, attack precision, binary accuracy,
    and the number of true attack nodes.
    """
    normal_id = resolve_normal_class_id(label_mapping, normal_label)
    prefer_norm = normalize_label_name(prefer_label) if prefer_label is not None else None
    eps = 1e-12

    candidates = []

    def scan(prefer_only: bool, min_attack_required: int):
        scanned = 0
        for raw_g in iter_raw_graphs(graph_folder, split):
            scanned += 1
            if scanned > max_scan:
                break

            label_name = get_graph_label_name(raw_g, label_mapping)
            if prefer_only and prefer_norm is not None and normalize_label_name(label_name) != prefer_norm:
                continue

            data = graph_dict_to_data(raw_g)
            if not hasattr(data, node_target):
                continue

            node_y = getattr(data, node_target).view(-1)
            if hasattr(data, 'node_mask'):
                node_mask = data.node_mask.view(-1).bool()
            else:
                node_mask = torch.ones_like(node_y, dtype=torch.bool)

            if node_target == 'node_is_attack':
                true_attack = (node_y == 1) & node_mask
                true_normal = (node_y == 0) & node_mask
            else:
                true_attack = (node_y != normal_id) & node_mask
                true_normal = (node_y == normal_id) & node_mask

            true_attack_count = int(true_attack.sum().item())
            valid_count = int(node_mask.sum().item())
            attack_ratio = true_attack_count / max(valid_count, 1)

            # mixed window requirement and attack-node range requirement
            if not bool(true_normal.any().item()):
                continue

            if true_attack_count < min_attack_required:
                continue

            # Avoid attack-dominant windows for cleaner visualization
            if max_true_attack_nodes is not None and true_attack_count > max_true_attack_nodes:
                continue

            if max_attack_ratio is not None and attack_ratio > max_attack_ratio:
                continue

            result = get_single_graph_node_classification(
                model=model,
                data=data,
                label_mapping=label_mapping,
                device=device,
                node_target=node_target,
                normal_label=normal_label,
            )
            metrics = _binary_node_metrics_from_result(result)

            graph_ok = bool(metrics['graph_correct']) or not require_graph_correct
            recall_ok = metrics['attack_recall'] >= 1.0 - eps
            precision_ok = metrics['attack_precision'] >= 1.0 - eps
            tp_ok = metrics['tp'] >= min_tp_attack_nodes
            acc_ok = metrics['binary_acc'] >= min_binary_node_acc

            strict_ok = bool(graph_ok and recall_ok and precision_ok and tp_ok and acc_ok)

            # Higher score is better. Strict cases always rank above fallback cases.
            score = (
                int(strict_ok),
                int(metrics['graph_correct']),
                metrics['attack_recall'],
                metrics['attack_precision'],
                metrics['binary_acc'],
                metrics['tp'],
                metrics['num_true_attack'],
                -metrics['fp'],
                -metrics['fn'],
                -abs(metrics['num_true_attack'] - metrics['num_pred_attack']),
            )

            candidates.append({
                'raw_graph': raw_g,
                'data': data,
                'label_name': label_name,
                'result': result,
                'metrics': metrics,
                'selection_status': 'strict_perfect_match' if strict_ok else 'best_fallback',
                'score': score,
                'attack_ratio': attack_ratio,
                'strict_ok': strict_ok,
                'min_attack_required': min_attack_required,
            })

    # Pass 1: preferred attack label, if available. Pass 2: any label.
    for prefer_only in ([True, False] if prefer_norm is not None else [False]):
        scan(prefer_only=prefer_only, min_attack_required=min_true_attack_nodes)

    if len(candidates) == 0:
        # Relaxed fallback: at least 2 true attack nodes if no graph with >=3 is found.
        relaxed_min_attack = max(2, min_tp_attack_nodes)
        print(
            f"No candidate found with true_attack_count >= {min_true_attack_nodes}. "
            f"Relaxing to true_attack_count >= {relaxed_min_attack}."
        )
        for prefer_only in ([True, False] if prefer_norm is not None else [False]):
            scan(prefer_only=prefer_only, min_attack_required=relaxed_min_attack)

    if len(candidates) == 0:
        return None

    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)
    best = candidates[0]

    # If the best was from the relaxed fallback, make the status explicit.
    if best['min_attack_required'] < min_true_attack_nodes and best['selection_status'] == 'best_fallback':
        best['selection_status'] = 'relaxed_best_fallback'
    elif best['min_attack_required'] < min_true_attack_nodes and best['selection_status'] == 'strict_perfect_match':
        best['selection_status'] = 'relaxed_strict_perfect_match'

    return best



# ============================================================
# Faster graph selection for Figure 2b
# ============================================================
# This optimized selector keeps the existing plotting code unchanged.
# It only changes how the attack/mixed graph is selected:
#   1) CPU pre-filter by ground-truth node labels;
#   2) support multiple preferred attack labels;
#   3) run the model only on a small number of candidates;
#   4) use batched inference instead of one-graph-at-a-time inference.

from torch_geometric.loader import DataLoader


def _normalize_prefer_labels(prefer_labels=None, prefer_label=None):
    """Accept either prefer_labels=[...] or old prefer_label='...' for backward compatibility."""
    labels = []
    if prefer_labels is not None:
        if isinstance(prefer_labels, str):
            labels = [prefer_labels]
        else:
            labels = list(prefer_labels)
    elif prefer_label is not None:
        labels = [prefer_label]

    labels = [str(x) for x in labels if x is not None and str(x).strip() != ""]
    return labels


def _graph_label_matches_preference(label_name: str, prefer_norm_list):
    if not prefer_norm_list:
        return True
    return normalize_label_name(label_name) in prefer_norm_list


def _true_attack_summary_from_data(
    data: Data,
    normal_id: int,
    node_target: str = "node_y",
):
    """Fast CPU-only summary from node labels. No model forward is needed."""
    if not hasattr(data, node_target):
        return None

    node_y = getattr(data, node_target).view(-1).detach().cpu().long()

    if hasattr(data, "node_mask"):
        node_mask = data.node_mask.view(-1).detach().cpu().bool()
    else:
        node_mask = torch.ones_like(node_y, dtype=torch.bool)

    valid_y = node_y[node_mask]
    if valid_y.numel() == 0:
        return None

    if node_target == "node_is_attack":
        true_attack = valid_y == 1
        true_normal = valid_y == 0
        attack_classes = [1] if bool(true_attack.any().item()) else []
    else:
        true_attack = valid_y != int(normal_id)
        true_normal = valid_y == int(normal_id)
        attack_classes = sorted(set(valid_y[true_attack].cpu().numpy().astype(int).tolist()))

    return {
        "valid_count": int(valid_y.numel()),
        "true_attack_count": int(true_attack.sum().item()),
        "true_normal_count": int(true_normal.sum().item()),
        "attack_ratio": float(true_attack.sum().item() / max(valid_y.numel(), 1)),
        "attack_classes": attack_classes,
        "num_attack_classes": len(attack_classes),
    }


def _prefilter_candidate_graphs_fast(
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    node_target: str,
    normal_label: str,
    prefer_labels=None,
    max_scan: int = 30000,
    min_true_attack_nodes: int = 3,
    max_true_attack_nodes: Optional[int] = 10,
    max_attack_ratio: Optional[float] = 0.20,
    min_true_attack_classes: int = 1,
    max_candidates_per_label: int = 250,
):
    """
    Fast CPU pre-filter.

    Returns a small list of candidate raw graphs/Data objects.
    This avoids running the model on the entire split.
    """
    normal_id = resolve_normal_class_id(label_mapping, normal_label)

    prefer_labels = _normalize_prefer_labels(prefer_labels=prefer_labels)
    prefer_norm_list = [normalize_label_name(x) for x in prefer_labels]

    # If prefer_labels is given, keep per-label quotas.
    per_label_counter = {normalize_label_name(x): 0 for x in prefer_labels}
    no_pref_counter = 0

    candidates = []
    scanned = 0

    for raw_g in iter_raw_graphs(graph_folder, split):
        scanned += 1
        if max_scan is not None and scanned > max_scan:
            break

        label_name = get_graph_label_name(raw_g, label_mapping)
        label_norm = normalize_label_name(label_name)

        # If multiple preferred labels are provided, skip labels outside the list.
        if prefer_norm_list and label_norm not in prefer_norm_list:
            continue

        # Per-label quota to avoid one frequent label dominating candidates.
        if prefer_norm_list:
            if per_label_counter.get(label_norm, 0) >= max_candidates_per_label:
                continue
        else:
            if no_pref_counter >= max_candidates_per_label:
                continue

        data = graph_dict_to_data(raw_g)
        summary = _true_attack_summary_from_data(
            data=data,
            normal_id=normal_id,
            node_target=node_target,
        )

        if summary is None:
            continue

        # Need a mixed graph: at least one normal node and at least some attack nodes.
        if summary["true_normal_count"] <= 0:
            continue

        if summary["true_attack_count"] < min_true_attack_nodes:
            continue

        if max_true_attack_nodes is not None and summary["true_attack_count"] > max_true_attack_nodes:
            continue

        if max_attack_ratio is not None and summary["attack_ratio"] > max_attack_ratio:
            continue

        if summary["num_attack_classes"] < min_true_attack_classes:
            continue

        # Store compact candidate
        if prefer_norm_list:
            per_label_counter[label_norm] = per_label_counter.get(label_norm, 0) + 1
            label_rank = prefer_norm_list.index(label_norm) if label_norm in prefer_norm_list else 10_000
        else:
            no_pref_counter += 1
            label_rank = 0

        candidates.append({
            "raw_graph": raw_g,
            "data": data,
            "label_name": label_name,
            "label_rank": label_rank,
            "summary": summary,
        })

        # Early stop when quotas are filled for all preferred labels.
        if prefer_norm_list and all(per_label_counter.get(x, 0) >= max_candidates_per_label for x in prefer_norm_list):
            break

    print("=" * 100)
    print("Fast CPU prefilter for Figure 2b")
    print(f"  scanned graphs             : {scanned}")
    print(f"  candidate graphs           : {len(candidates)}")
    print(f"  preferred labels           : {prefer_labels if prefer_labels else 'any attack label'}")
    if prefer_norm_list:
        print(f"  candidates per label       : {per_label_counter}")
    print(f"  true_attack_count range    : [{min_true_attack_nodes}, {max_true_attack_nodes}]")
    print(f"  max_attack_ratio           : {max_attack_ratio}")
    print(f"  min_true_attack_classes    : {min_true_attack_classes}")
    print("=" * 100)

    return candidates

from torch_geometric.data import Data
def clean_data_for_model_batch(data):
    """
    Keep only tensor attributes needed by the model and node visualization.
    This avoids PyG DataLoader errors caused by non-uniform dict metadata,
    e.g., KeyError: 'combined'.
    """
    clean = Data()

    # Required graph fields
    clean.x = data.x.float()
    clean.edge_index = data.edge_index.long()

    if hasattr(data, "edge_attr") and data.edge_attr is not None:
        clean.edge_attr = data.edge_attr.float()

    if hasattr(data, "edge_type") and data.edge_type is not None:
        clean.edge_type = data.edge_type.long()

    # ID token field
    if hasattr(data, "id_token") and data.id_token is not None:
        clean.id_token = data.id_token.long()
    elif hasattr(data, "id_index") and data.id_index is not None:
        clean.id_token = data.id_index.long()
    else:
        raise AttributeError("Data must contain id_token or id_index.")

    # Graph label
    clean.y = data.y.view(-1).long()

    # Node labels
    if hasattr(data, "node_y") and data.node_y is not None:
        clean.node_y = data.node_y.long()

    if hasattr(data, "node_mask") and data.node_mask is not None:
        clean.node_mask = data.node_mask.bool()

    if hasattr(data, "node_is_attack") and data.node_is_attack is not None:
        clean.node_is_attack = data.node_is_attack.long()

    return clean

@torch.no_grad()
def _evaluate_candidate_batch_for_node_selection(
    model,
    candidate_batch,
    label_mapping: Dict[int, str],
    device: str,
    node_target: str,
    normal_label: str,
):
    """
    Run one batch of candidate graphs and return records compatible with
    the existing notebook variables:
        raw_graph, data, label_name, result, metrics, selection_status, score, attack_ratio
    """
    if len(candidate_batch) == 0:
        return []

    normal_id = resolve_normal_class_id(label_mapping, normal_label)

    data_list = [
        clean_data_for_model_batch(x["data"])
        for x in candidate_batch
    ]

    loader = DataLoader(data_list, batch_size=len(data_list), shuffle=False)
    batch = next(iter(loader)).to(device)

    batch.x = batch.x.float()
    batch.edge_index = batch.edge_index.long()
    if hasattr(batch, "edge_attr"):
        batch.edge_attr = batch.edge_attr.float()
    if hasattr(batch, "edge_type"):
        batch.edge_type = batch.edge_type.long()
    if hasattr(batch, "id_token"):
        batch.id_token = batch.id_token.long()

    output = model(batch, return_node_logits=True)
    graph_logits, node_logits = parse_multitask_output(output)

    if graph_logits is None:
        raise RuntimeError("Model did not return graph_logits.")
    if node_logits is None:
        raise RuntimeError("Model did not return node_logits. Check num_node_classes and checkpoint loading.")

    graph_prob = F.softmax(graph_logits, dim=1).detach().cpu()
    graph_pred = graph_prob.argmax(dim=1).detach().cpu().long()

    node_prob = F.softmax(node_logits, dim=1).detach().cpu()
    node_pred = node_prob.argmax(dim=1).detach().cpu().long()
    node_conf = node_prob.max(dim=1).values.detach().cpu()

    if node_prob.size(1) == 2 and node_target == "node_is_attack":
        attack_prob = node_prob[:, 1]
        pred_is_attack_all = node_pred == 1
    else:
        attack_prob = 1.0 - node_prob[:, normal_id]
        pred_is_attack_all = node_pred != normal_id

    node_true_all = getattr(batch, node_target).view(-1).detach().cpu().long()
    if hasattr(batch, "node_mask"):
        node_mask_all = batch.node_mask.view(-1).detach().cpu().bool()
    else:
        node_mask_all = torch.ones_like(node_true_all, dtype=torch.bool)

    batch_vec = batch.batch.detach().cpu().long()
    graph_true_all = batch.y.view(-1).detach().cpu().long()

    records = []

    for local_i, cand in enumerate(candidate_batch):
        node_sel = batch_vec == local_i
        data_i = cand["data"]

        node_true_i = node_true_all[node_sel].clone()
        node_pred_i = node_pred[node_sel].clone()
        node_prob_i = node_prob[node_sel].clone()
        node_conf_i = node_conf[node_sel].clone()
        attack_prob_i = attack_prob[node_sel].clone()
        node_mask_i = node_mask_all[node_sel].clone()

        if node_target == "node_is_attack":
            true_is_attack_i = node_true_i == 1
            pred_is_attack_i = node_pred_i == 1
        else:
            true_is_attack_i = node_true_i != int(normal_id)
            pred_is_attack_i = node_pred_i != int(normal_id)

        valid = node_mask_i.numpy().astype(bool)
        true_bin = true_is_attack_i.numpy()[valid].astype(int)
        pred_bin = pred_is_attack_i.numpy()[valid].astype(int)
        bin_acc = float((true_bin == pred_bin).mean()) if valid.sum() > 0 else 0.0

        graph_true_idx = int(graph_true_all[local_i].item())
        graph_pred_idx = int(graph_pred[local_i].item())

        result = {
            "graph_id": getattr(data_i, "graph_id", "unknown_graph"),
            "window_label": getattr(data_i, "window_label", None),
            "source_class": getattr(data_i, "source_class", None),
            "graph_true_idx": graph_true_idx,
            "graph_pred_idx": graph_pred_idx,
            "graph_true_name": label_mapping.get(graph_true_idx, f"class_{graph_true_idx}"),
            "graph_pred_name": label_mapping.get(graph_pred_idx, f"class_{graph_pred_idx}"),
            "graph_prob": graph_prob[local_i:local_i+1].clone(),
            "node_true": node_true_i,
            "node_pred": node_pred_i,
            "node_prob": node_prob_i,
            "node_conf": node_conf_i,
            "attack_prob": attack_prob_i,
            "node_mask": node_mask_i,
            "true_is_attack": true_is_attack_i.cpu().bool(),
            "pred_is_attack": pred_is_attack_i.cpu().bool(),
            "normal_class_id": normal_id,
            "normal_label": label_mapping.get(normal_id, str(normal_label)),
            "binary_attack_acc": bin_acc,
            "num_attack_true": int(true_is_attack_i[node_mask_i].sum().item()),
            "num_attack_pred": int(pred_is_attack_i[node_mask_i].sum().item()),
            "num_nodes": int(node_mask_i.sum().item()),
        }

        metrics = _binary_node_metrics_from_result(result)

        # Add multiclass node accuracy because this figure can visualize multiclass predictions.
        node_true_np = node_true_i.numpy()[valid]
        node_pred_np = node_pred_i.numpy()[valid]
        metrics["multiclass_acc"] = float((node_true_np == node_pred_np).mean()) if valid.sum() > 0 else 0.0

        true_attack_classes = sorted(set(node_true_i[true_is_attack_i & node_mask_i].numpy().astype(int).tolist()))
        pred_attack_classes = sorted(set(node_pred_i[pred_is_attack_i & node_mask_i].numpy().astype(int).tolist()))
        metrics["true_attack_classes"] = true_attack_classes
        metrics["pred_attack_classes"] = pred_attack_classes
        metrics["num_true_attack_classes"] = len(true_attack_classes)
        metrics["num_pred_attack_classes"] = len(pred_attack_classes)

        attack_ratio = metrics["num_true_attack"] / max(metrics["num_nodes"], 1)

        # Higher is better.
        # label_rank is inverted so labels earlier in PREFERRED_ATTACK_LABELS are preferred.
        score = (
            int(metrics["graph_correct"]),
            metrics["attack_recall"],
            metrics["attack_precision"],
            metrics["binary_acc"],
            metrics["multiclass_acc"],
            metrics["tp"],
            -cand.get("label_rank", 0),
            -abs(metrics["num_true_attack"] - 5),  # prefer visually moderate cases around 5 attack nodes
            -metrics["fp"],
            -metrics["fn"],
        )

        records.append({
            "raw_graph": cand["raw_graph"],
            "data": data_i,
            "label_name": cand["label_name"],
            "result": result,
            "metrics": metrics,
            "selection_status": "candidate_evaluated",
            "score": score,
            "attack_ratio": attack_ratio,
            "strict_ok": False,
            "label_rank": cand.get("label_rank", 0),
        })

    return records


def find_successful_node_classification_graph_fast(
    model,
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    device: str = "cpu",
    node_target: str = "node_y",
    normal_label: str = "normal",
    prefer_labels=None,
    prefer_label: Optional[str] = None,
    max_scan: int = 30000,
    min_true_attack_nodes: int = 3,
    max_true_attack_nodes: Optional[int] = 10,
    min_tp_attack_nodes: int = 2,
    min_binary_node_acc: float = 0.99,
    max_attack_ratio: Optional[float] = 0.20,
    min_true_attack_classes: int = 1,
    require_graph_correct: bool = True,
    batch_size: int = 256,
    max_candidates_per_label: int = 250,
    early_stop_after_strict: bool = True,
):
    """
    Optimized selector for Figure 2b.

    It can search across multiple attack labels, e.g.
        prefer_labels=['fuzzing', 'systematic', 'speed', 'combined', 'interval', 'dos']

    The existing plotting code can stay unchanged because the returned object
    has the same structure as the old selector.
    """
    prefer_labels = _normalize_prefer_labels(prefer_labels=prefer_labels, prefer_label=prefer_label)
    eps = 1e-12

    candidates = _prefilter_candidate_graphs_fast(
        graph_folder=graph_folder,
        split=split,
        label_mapping=label_mapping,
        node_target=node_target,
        normal_label=normal_label,
        prefer_labels=prefer_labels,
        max_scan=max_scan,
        min_true_attack_nodes=min_true_attack_nodes,
        max_true_attack_nodes=max_true_attack_nodes,
        max_attack_ratio=max_attack_ratio,
        min_true_attack_classes=min_true_attack_classes,
        max_candidates_per_label=max_candidates_per_label,
    )

    if len(candidates) == 0 and min_true_attack_classes > 1:
        print(
            f"No candidate found with at least {min_true_attack_classes} attack classes. "
            "Relaxing min_true_attack_classes to 1."
        )
        candidates = _prefilter_candidate_graphs_fast(
            graph_folder=graph_folder,
            split=split,
            label_mapping=label_mapping,
            node_target=node_target,
            normal_label=normal_label,
            prefer_labels=prefer_labels,
            max_scan=max_scan,
            min_true_attack_nodes=min_true_attack_nodes,
            max_true_attack_nodes=max_true_attack_nodes,
            max_attack_ratio=max_attack_ratio,
            min_true_attack_classes=1,
            max_candidates_per_label=max_candidates_per_label,
        )

    if len(candidates) == 0:
        print("No candidate found after CPU prefilter.")
        return None

    model.eval()

    all_records = []
    strict_records = []

    for start in range(0, len(candidates), batch_size):
        cand_batch = candidates[start:start + batch_size]

        batch_records = _evaluate_candidate_batch_for_node_selection(
            model=model,
            candidate_batch=cand_batch,
            label_mapping=label_mapping,
            device=device,
            node_target=node_target,
            normal_label=normal_label,
        )

        for rec in batch_records:
            m = rec["metrics"]

            graph_ok = bool(m["graph_correct"]) or not require_graph_correct
            recall_ok = m["attack_recall"] >= 1.0 - eps
            precision_ok = m["attack_precision"] >= 1.0 - eps
            tp_ok = m["tp"] >= min_tp_attack_nodes
            acc_ok = m["binary_acc"] >= min_binary_node_acc

            strict_ok = bool(graph_ok and recall_ok and precision_ok and tp_ok and acc_ok)
            rec["strict_ok"] = strict_ok
            rec["selection_status"] = "strict_perfect_match" if strict_ok else "best_fallback"

            all_records.append(rec)
            if strict_ok:
                strict_records.append(rec)

        if early_stop_after_strict and len(strict_records) > 0:
            break

    if len(all_records) == 0:
        return None

    if len(strict_records) > 0:
        best = sorted(strict_records, key=lambda x: x["score"], reverse=True)[0]
        best["selection_status"] = "strict_perfect_match"
    else:
        best = sorted(all_records, key=lambda x: x["score"], reverse=True)[0]
        best["selection_status"] = "best_fallback"

    return best


def find_imperfect_node_classification_graph_fast(
    model,
    graph_folder: Path,
    split: str,
    label_mapping: Dict[int, str],
    device: str = "cpu",
    node_target: str = "node_y",
    normal_label: str = "normal",
    prefer_labels=None,
    prefer_label: Optional[str] = None,
    max_scan: int = 30000,
    min_true_attack_nodes: int = 3,
    max_true_attack_nodes: Optional[int] = 8,
    min_tp_attack_nodes: int = 2,

    # error control
    error_type: str = "fn",   # "fn", "fp", or "any"
    min_fn_nodes: int = 1,
    min_fp_nodes: int = 0,

    # keep graph reasonable: not perfect, not too bad
    min_binary_node_acc: float = 0.90,
    max_binary_node_acc: float = 0.99,
    max_attack_ratio: Optional[float] = 0.20,
    min_true_attack_classes: int = 1,

    # usually better for explainability: graph-level classification is correct,
    # but node-level localization has some mistakes.
    require_graph_correct: bool = True,

    batch_size: int = 256,
    max_candidates_per_label: int = 250,
    early_stop_after_match: bool = True,
):
    """
    Select an informative mixed graph/window where node classification is imperfect.

    The target case is useful for failure-case visualization:
      - the window contains both normal and attack nodes;
      - some attack nodes are correctly detected: TP >= min_tp_attack_nodes;
      - at least one node is misclassified, preferably an FN when error_type='fn';
      - binary node accuracy remains reasonably high so the figure is not a completely failed case;
      - graph-level prediction can be required to be correct.

    error_type:
        "fn"  -> require missed attack nodes: FN >= min_fn_nodes
        "fp"  -> require false alarms: FP >= min_fp_nodes
        "any" -> require at least one binary error: FN + FP >= 1

    Return object has the same structure as find_successful_node_classification_graph_fast,
    so the existing plotting functions can be reused without modification.
    """
    prefer_labels = _normalize_prefer_labels(prefer_labels=prefer_labels, prefer_label=prefer_label)
    error_type = str(error_type).lower().strip()
    if error_type not in {"fn", "fp", "any"}:
        raise ValueError("error_type must be one of: 'fn', 'fp', 'any'")

    candidates = _prefilter_candidate_graphs_fast(
        graph_folder=graph_folder,
        split=split,
        label_mapping=label_mapping,
        node_target=node_target,
        normal_label=normal_label,
        prefer_labels=prefer_labels,
        max_scan=max_scan,
        min_true_attack_nodes=min_true_attack_nodes,
        max_true_attack_nodes=max_true_attack_nodes,
        max_attack_ratio=max_attack_ratio,
        min_true_attack_classes=min_true_attack_classes,
        max_candidates_per_label=max_candidates_per_label,
    )

    if len(candidates) == 0 and min_true_attack_classes > 1:
        print(
            f"No imperfect-case candidate found with at least {min_true_attack_classes} attack classes. "
            "Relaxing min_true_attack_classes to 1."
        )
        candidates = _prefilter_candidate_graphs_fast(
            graph_folder=graph_folder,
            split=split,
            label_mapping=label_mapping,
            node_target=node_target,
            normal_label=normal_label,
            prefer_labels=prefer_labels,
            max_scan=max_scan,
            min_true_attack_nodes=min_true_attack_nodes,
            max_true_attack_nodes=max_true_attack_nodes,
            max_attack_ratio=max_attack_ratio,
            min_true_attack_classes=1,
            max_candidates_per_label=max_candidates_per_label,
        )

    if len(candidates) == 0:
        print("No imperfect-case candidate found after CPU prefilter.")
        return None

    model.eval()

    all_records = []
    matched_records = []

    for start in range(0, len(candidates), batch_size):
        cand_batch = candidates[start:start + batch_size]

        batch_records = _evaluate_candidate_batch_for_node_selection(
            model=model,
            candidate_batch=cand_batch,
            label_mapping=label_mapping,
            device=device,
            node_target=node_target,
            normal_label=normal_label,
        )

        for rec in batch_records:
            m = rec["metrics"]

            graph_ok = bool(m["graph_correct"]) or not require_graph_correct
            tp_ok = int(m["tp"]) >= int(min_tp_attack_nodes)
            acc_ok = (float(m["binary_acc"]) >= float(min_binary_node_acc)) and (float(m["binary_acc"]) <= float(max_binary_node_acc))
            imperfect_ok = not bool(m["exact_binary_match"])

            if error_type == "fn":
                error_ok = int(m["fn"]) >= int(min_fn_nodes)
            elif error_type == "fp":
                error_ok = int(m["fp"]) >= int(min_fp_nodes)
            else:
                error_ok = int(m["fn"] + m["fp"]) >= 1

            matched = bool(graph_ok and tp_ok and acc_ok and imperfect_ok and error_ok)
            rec["matched"] = matched
            rec["selection_status"] = "imperfect_match" if matched else "best_imperfect_fallback"

            # Ranking preference:
            # 1) graph-level prediction is correct;
            # 2) desired error exists;
            # 3) some attack nodes are still correctly detected;
            # 4) only a small number of node errors, preferably 1-2;
            # 5) node-level accuracy remains high.
            total_errors = int(m["fp"] + m["fn"])
            desired_errors = 1 if error_type in {"fn", "fp"} else 2
            desired_error_count = int(m["fn"] if error_type == "fn" else m["fp"] if error_type == "fp" else total_errors)

            ranking_score = (
                int(graph_ok),
                int(error_ok),
                int(tp_ok),
                -abs(total_errors - desired_errors),
                desired_error_count,
                int(m["tp"]),
                float(m["binary_acc"]),
                float(m["attack_recall"]),
                float(m["attack_precision"]),
                -int(m["fp"]),
                -int(m["fn"]),
                -int(rec.get("label_rank", 0)),
            )
            rec["ranking_score"] = ranking_score

            all_records.append(rec)
            if matched:
                matched_records.append(rec)

        if early_stop_after_match and len(matched_records) > 0:
            break

    if len(all_records) == 0:
        print("No imperfect-case record could be evaluated.")
        return None

    if len(matched_records) > 0:
        best = sorted(matched_records, key=lambda x: x["ranking_score"], reverse=True)[0]
        best["selection_status"] = "imperfect_match"
    else:
        best = sorted(all_records, key=lambda x: x["ranking_score"], reverse=True)[0]
        best["selection_status"] = "best_imperfect_fallback"

    return best



In [ ]:
# ------------------------------------------------------------
# Select graphs for node-classification visualization
# ------------------------------------------------------------
normal_class_id = resolve_normal_class_id(ckpt_label_mapping, resolved_normal_label)
node_target_for_vis = ckpt.get('args', {}).get('node_target', 'node_y')

print('normal_class_id =', normal_class_id, '| normal label =', ckpt_label_mapping.get(normal_class_id))
print('node_target_for_vis =', node_target_for_vis)

# ------------------------------------------------------------
# Faster and more robust selection for Figure 2b
# ------------------------------------------------------------
# Recommended target classes for visualization:
#   - fuzzing/systematic/speed/combined: more informative than DoS because they are less saturated;
#   - interval/dos: easier fallback classes with high node-level performance;
#   - gear/rpm: included later as fallback, but they are harder according to the node report.
#
# The selector searches across these labels instead of using only the single
# attack_label_name selected earlier in the notebook.
PREFERRED_ATTACK_LABELS_FOR_VIS = [
    'fuzzing',
    'systematic',
    'speed',
    'combined',
    'interval',
    'dos',
    'gear',
    'rpm',
]

# Strict target:
#   - 3 <= true_attack_count <= 10;
#   - at least 2 correctly detected attack nodes;
#   - graph prediction is correct;
#   - attack recall = 1.0;
#   - attack precision = 1.0;
#   - binary node accuracy is close to 1.0.
MIN_TRUE_ATTACK_NODES = 3
MAX_TRUE_ATTACK_NODES = 5
MIN_TP_ATTACK_NODES = 2
MIN_BINARY_NODE_ACC = 0.99
MAX_ATTACK_RATIO = 0.20

# Usually each generated window contains one dominant attack type.
# Keep this as 1 for fast/reliable selection.
# If you specifically want a graph containing multiple attack classes,
# change this to 2, but it may be much slower or may not find a strict case.
MIN_TRUE_ATTACK_CLASSES = 1

selected_success = find_successful_node_classification_graph_fast(
    model=model,
    graph_folder=GRAPH_FOLDER,
    split=SPLIT,
    label_mapping=ckpt_label_mapping,
    device=DEVICE,
    node_target=node_target_for_vis,
    normal_label=resolved_normal_label,
    prefer_labels=PREFERRED_ATTACK_LABELS_FOR_VIS,
    max_scan=30000,
    min_true_attack_nodes=MIN_TRUE_ATTACK_NODES,
    max_true_attack_nodes=MAX_TRUE_ATTACK_NODES,
    min_tp_attack_nodes=MIN_TP_ATTACK_NODES,
    min_binary_node_acc=MIN_BINARY_NODE_ACC,
    max_attack_ratio=MAX_ATTACK_RATIO,
    min_true_attack_classes=MIN_TRUE_ATTACK_CLASSES,
    require_graph_correct=True,
    batch_size=256,
    max_candidates_per_label=150,
    early_stop_after_strict=True,
)

if selected_success is None:
    print('No suitable mixed graph found. Falling back to the previously selected attack graph.')
    raw_node_attack, data_node_attack, node_attack_label_name = raw_attack, data_attack, attack_label_name
    node_cls_attack = get_single_graph_node_classification(
        model=model,
        data=data_node_attack,
        label_mapping=ckpt_label_mapping,
        device=DEVICE,
        node_target=node_target_for_vis,
        normal_label=resolved_normal_label,
    )
    selection_status = 'fallback_original_attack_graph'
    selection_metrics = _binary_node_metrics_from_result(node_cls_attack)
    attack_ratio_selected = selection_metrics['num_true_attack'] / max(selection_metrics['num_nodes'], 1)
else:
    raw_node_attack = selected_success['raw_graph']
    data_node_attack = selected_success['data']
    node_attack_label_name = selected_success['label_name']
    node_cls_attack = selected_success['result']
    selection_status = selected_success['selection_status']
    selection_metrics = selected_success['metrics']
    attack_ratio_selected = selected_success.get(
        'attack_ratio',
        selection_metrics['num_true_attack'] / max(selection_metrics['num_nodes'], 1)
    )


def _class_ids_to_names(ids, label_mapping):
    return [label_mapping.get(int(i), f'class_{int(i)}') for i in ids]


print('\nSelected graph for Figure 2b')
print('  status              :', selection_status)
print('  graph_id            :', data_node_attack.graph_id)
print('  graph label         :', node_attack_label_name)
print('  preferred labels    :', PREFERRED_ATTACK_LABELS_FOR_VIS)
print('  graph true/pred     :', node_cls_attack['graph_true_name'], '->', node_cls_attack['graph_pred_name'])
print('  graph_correct       :', selection_metrics['graph_correct'])
print(
    '  true attack nodes   :',
    selection_metrics['num_true_attack'],
    f'(required {MIN_TRUE_ATTACK_NODES} <= count <= {MAX_TRUE_ATTACK_NODES})'
)
print('  pred attack nodes   :', selection_metrics['num_pred_attack'])
print('  attack ratio        :', f"{attack_ratio_selected:.4f}", f'(target <= {MAX_ATTACK_RATIO})')
print(
    '  true attack classes :',
    _class_ids_to_names(selection_metrics.get('true_attack_classes', []), ckpt_label_mapping)
)
print(
    '  pred attack classes :',
    _class_ids_to_names(selection_metrics.get('pred_attack_classes', []), ckpt_label_mapping)
)
print('  TP / FP / FN / TN   :', selection_metrics['tp'], '/', selection_metrics['fp'], '/', selection_metrics['fn'], '/', selection_metrics['tn'])
print('  TP attack nodes     :', selection_metrics['tp'], f'(required >= {MIN_TP_ATTACK_NODES})')
print('  binary node acc     :', f"{selection_metrics['binary_acc']:.4f}", f'(target >= {MIN_BINARY_NODE_ACC})')
print('  attack recall       :', f"{selection_metrics['attack_recall']:.4f}", '(target = 1.0)')
print('  attack precision    :', f"{selection_metrics['attack_precision']:.4f}", '(target = 1.0)')
if 'multiclass_acc' in selection_metrics:
    print('  multiclass node acc :', f"{selection_metrics['multiclass_acc']:.4f}")

if selection_status not in {'strict_perfect_match', 'relaxed_strict_perfect_match'}:
    print('\n[Warning] A strict perfect case was not found. The selected graph is the best fallback under the ranking criteria.')



# ------------------------------------------------------------
# Additional selection: imperfect mixed graph for failure-case visualization
# ------------------------------------------------------------
# This case is useful for showing that the node head detects several attack
# nodes correctly but still misses or misclassifies a small number of nodes.
# It is better for qualitative failure analysis than showing only perfect cases.
PREFERRED_ATTACK_LABELS_FOR_ERROR_VIS = [
    'fuzzing',
    'systematic',
    'speed',
    'combined',
    'interval',
    'gear',
    'rpm',
    'dos',
]

ERROR_VIS_TYPE = 'fn'  # 'fn' = missed attack node, 'fp' = false alarm, 'any' = any binary node error

selected_error = find_imperfect_node_classification_graph_fast(
    model=model,
    graph_folder=GRAPH_FOLDER,
    split=SPLIT,
    label_mapping=ckpt_label_mapping,
    device=DEVICE,
    node_target=node_target_for_vis,
    normal_label=resolved_normal_label,
    prefer_labels=PREFERRED_ATTACK_LABELS_FOR_ERROR_VIS,
    max_scan=30000,
    min_true_attack_nodes=3,
    max_true_attack_nodes=8,
    min_tp_attack_nodes=2,
    error_type=ERROR_VIS_TYPE,
    min_fn_nodes=1,
    min_fp_nodes=1 if ERROR_VIS_TYPE == 'fp' else 0,
    min_binary_node_acc=0.90,
    max_binary_node_acc=0.99,
    max_attack_ratio=0.20,
    min_true_attack_classes=1,
    require_graph_correct=True,
    batch_size=256,
    max_candidates_per_label=150,
    early_stop_after_match=True,
)

if selected_error is None:
    print('\nNo suitable imperfect mixed graph found. The imperfect-case figures will be skipped.')
    raw_node_error = None
    data_node_error = None
    node_error_label_name = None
    node_cls_error = None
    error_selection_metrics = None
else:
    raw_node_error = selected_error['raw_graph']
    data_node_error = selected_error['data']
    node_error_label_name = selected_error['label_name']
    node_cls_error = selected_error['result']
    error_selection_metrics = selected_error['metrics']

    print('\nSelected imperfect graph for node-level failure-case visualization')
    print('  status              :', selected_error['selection_status'])
    print('  graph_id            :', data_node_error.graph_id)
    print('  graph label         :', node_error_label_name)
    print('  preferred labels    :', PREFERRED_ATTACK_LABELS_FOR_ERROR_VIS)
    print('  graph true/pred     :', node_cls_error['graph_true_name'], '->', node_cls_error['graph_pred_name'])
    print('  graph_correct       :', error_selection_metrics['graph_correct'])
    print('  true attack nodes   :', error_selection_metrics['num_true_attack'])
    print('  pred attack nodes   :', error_selection_metrics['num_pred_attack'])
    print('  true attack classes :', _class_ids_to_names(error_selection_metrics.get('true_attack_classes', []), ckpt_label_mapping))
    print('  pred attack classes :', _class_ids_to_names(error_selection_metrics.get('pred_attack_classes', []), ckpt_label_mapping))
    print('  TP / FP / FN / TN   :', error_selection_metrics['tp'], '/', error_selection_metrics['fp'], '/', error_selection_metrics['fn'], '/', error_selection_metrics['tn'])
    print('  binary node acc     :', f"{error_selection_metrics['binary_acc']:.4f}")
    print('  attack recall       :', f"{error_selection_metrics['attack_recall']:.4f}")
    print('  attack precision    :', f"{error_selection_metrics['attack_precision']:.4f}")
    if 'multiclass_acc' in error_selection_metrics:
        print('  multiclass node acc :', f"{error_selection_metrics['multiclass_acc']:.4f}")

summarize_node_classification(node_cls_attack, name='selected successful attack/mixed case')
if node_cls_error is not None:
    summarize_node_classification(node_cls_error, name='selected imperfect attack/mixed case')

# Save detailed per-node prediction tables for the two selected graph-layout cases
node_cls_dir = Path('../figures/explainability')
node_cls_dir.mkdir(parents=True, exist_ok=True)

node_attack_df = node_classification_result_to_df(node_cls_attack, data_node_attack, ckpt_label_mapping)
if node_cls_error is not None:
    node_error_df = node_classification_result_to_df(node_cls_error, data_node_error, ckpt_label_mapping)
else:
    node_error_df = None

node_attack_df.to_csv(node_cls_dir / 'node_classification_correct_case.csv', index=False)
if node_error_df is not None:
    node_error_df.to_csv(node_cls_dir / 'node_classification_imperfect_case.csv', index=False)

print('[*] Saved node prediction CSV files:')
print('   ', node_cls_dir / 'node_classification_correct_case.csv')
if node_error_df is not None:
    print('   ', node_cls_dir / 'node_classification_imperfect_case.csv')

display(node_attack_df)
if node_error_df is not None:
    display(node_error_df)


In [ ]:

# ------------------------------------------------------------
# Graph layout visualization for node classification
# ------------------------------------------------------------
from typing import Dict, Any, Optional, Tuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError as e:
    raise ImportError(
        "networkx is required for graph layout visualization. "
        "Install it with: pip install networkx"
    ) from e


RELATION_TYPE_NAMES = {
    0: "Temporal",
    1: "Same-ID",
    2: "Payload-sim",
    3: "Timing-aff",
}

RELATION_TYPE_COLORS = {
    0: "#9E9E9E",
    1: "#59A14F",
    2: "#F28E2B",
    3: "#B07AA1",
}

NODE_NORMAL_COLOR = "#4C78A8"
NODE_ATTACK_COLOR = "#E45756"


def _get_edge_arrays_from_data(data: Data) -> Tuple[np.ndarray, np.ndarray]:
    """Return edge_index [2, E] and edge_type [E] from a PyG Data object."""
    edge_index = data.edge_index.detach().cpu().numpy()
    if hasattr(data, "edge_type"):
        edge_type = data.edge_type.detach().cpu().numpy()
    else:
        edge_type = np.zeros(edge_index.shape[1], dtype=int)
    return edge_index, edge_type


def _build_layout_graph(
    data: Data,
    node_mask: Optional[np.ndarray] = None,
    make_undirected: bool = True,
):
    """Build a NetworkX graph for layout computation."""
    num_nodes = int(data.x.size(0))
    if node_mask is None:
        valid_nodes = np.arange(num_nodes)
    else:
        valid_nodes = np.where(node_mask.astype(bool))[0]

    G = nx.Graph() if make_undirected else nx.DiGraph()
    G.add_nodes_from(valid_nodes.tolist())

    edge_index, _ = _get_edge_arrays_from_data(data)
    valid_set = set(valid_nodes.tolist())

    for u, v in edge_index.T:
        u, v = int(u), int(v)
        if u in valid_set and v in valid_set:
            G.add_edge(u, v)

    return G, valid_nodes


def _sample_edges_by_relation(
    edge_index: np.ndarray,
    edge_type: np.ndarray,
    valid_nodes: np.ndarray,
    max_edges_per_relation: Optional[int] = 350,
    seed: int = 42,
):
    """Group edges by relation type and optionally subsample for readable plots."""
    rng = np.random.default_rng(seed)
    valid_set = set(valid_nodes.tolist())
    rel_edges: Dict[int, list] = {}

    for k, (u, v) in enumerate(edge_index.T):
        u, v = int(u), int(v)
        if u not in valid_set or v not in valid_set:
            continue
        r = int(edge_type[k]) if k < len(edge_type) else 0
        rel_edges.setdefault(r, []).append((u, v))

    if max_edges_per_relation is not None:
        for r, edges in list(rel_edges.items()):
            if len(edges) > max_edges_per_relation:
                idx = rng.choice(len(edges), size=max_edges_per_relation, replace=False)
                rel_edges[r] = [edges[i] for i in idx]

    return rel_edges


def _draw_relation_edges(
    ax,
    G_layout,
    pos,
    data: Data,
    valid_nodes: np.ndarray,
    draw_relation_edges: bool = True,
    max_edges_per_relation: Optional[int] = 350,
    edge_alpha: float = 0.22,
):
    """Draw graph edges, optionally colored by relation type."""
    edge_index, edge_type = _get_edge_arrays_from_data(data)

    if draw_relation_edges:
        rel_edges = _sample_edges_by_relation(
            edge_index=edge_index,
            edge_type=edge_type,
            valid_nodes=valid_nodes,
            max_edges_per_relation=max_edges_per_relation,
        )
        for r, edges in sorted(rel_edges.items(), key=lambda kv: kv[0]):
            if not edges:
                continue
            nx.draw_networkx_edges(
                G_layout,
                pos,
                edgelist=edges,
                ax=ax,
                edge_color=RELATION_TYPE_COLORS.get(int(r), "#BDBDBD"),
                width=0.7,
                alpha=edge_alpha,
                arrows=False,
            )
    else:
        nx.draw_networkx_edges(
            G_layout,
            pos,
            ax=ax,
            edge_color="#BDBDBD",
            width=0.7,
            alpha=edge_alpha,
            arrows=False,
        )


def plot_graph_layout_node_classification(
    data: Data,
    result: Dict[str, Any],
    title: Optional[str] = None,
    save_path: Optional[str] = None,
    figsize=(14, 6.2),
    layout_seed: int = 42,
    draw_relation_edges: bool = True,
    max_edges_per_relation: Optional[int] = 350,
    show_labels_for_attack_and_error: bool = True,
    node_size: int = 180,
):
    """
    Draw graph layout with two panels:
        left  = ground-truth normal/attack node status
        right = predicted normal/attack node status

    This complements the timeline plot by showing how detected attack nodes
    are located within the graph structure.
    """
    true_bin = result["true_is_attack"].detach().cpu().numpy().astype(int)
    pred_bin = result["pred_is_attack"].detach().cpu().numpy().astype(int)
    node_mask = result["node_mask"].detach().cpu().numpy().astype(bool)

    G_layout, valid_nodes = _build_layout_graph(data, node_mask=node_mask, make_undirected=True)

    if G_layout.number_of_nodes() == 0:
        raise RuntimeError("Cannot draw graph layout: no valid nodes found.")

    # Stable layout for both panels.
    # k is chosen to reduce overlap for 64-node windows.
    pos = nx.spring_layout(
        G_layout,
        seed=layout_seed,
        k=1.25 / np.sqrt(max(G_layout.number_of_nodes(), 1)),
        iterations=300,
    )

    plt.rcParams.update({
        "axes.labelsize": 11,
        "font.size": 11,
        "legend.fontsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    statuses = [true_bin, pred_bin]
    panel_titles = ["Ground truth node status", "Predicted node status"]
    mismatch = true_bin != pred_bin

    for ax, status, panel_title in zip(axes, statuses, panel_titles):
        _draw_relation_edges(
            ax=ax,
            G_layout=G_layout,
            pos=pos,
            data=data,
            valid_nodes=valid_nodes,
            draw_relation_edges=draw_relation_edges,
            max_edges_per_relation=max_edges_per_relation,
            edge_alpha=0.22,
        )

        node_colors = [NODE_ATTACK_COLOR if status[i] == 1 else NODE_NORMAL_COLOR for i in valid_nodes]
        node_edgecolors = ["black" if mismatch[i] else "white" for i in valid_nodes]
        node_linewidths = [1.8 if mismatch[i] else 0.55 for i in valid_nodes]

        nx.draw_networkx_nodes(
            G_layout,
            pos,
            nodelist=valid_nodes.tolist(),
            node_color=node_colors,
            edgecolors=node_edgecolors,
            linewidths=node_linewidths,
            node_size=node_size,
            ax=ax,
        )

        if show_labels_for_attack_and_error:
            label_nodes = [int(i) for i in valid_nodes if true_bin[i] == 1 or pred_bin[i] == 1 or mismatch[i]]
            labels = {i: str(i) for i in label_nodes}
            nx.draw_networkx_labels(
                G_layout,
                pos,
                labels=labels,
                font_size=7,
                font_color="black",
                ax=ax,
            )

        ax.set_title(panel_title, fontweight="bold")
        ax.set_axis_off()

    if title is None:
        title = (
            f"Graph layout: {result['graph_id']} | "
            f"true={result['graph_true_name']} | pred={result['graph_pred_name']} | "
            f"node acc={result['binary_attack_acc']:.3f}"
        )
    fig.suptitle(title, fontweight="bold", y=1.02)

    node_legend = [
        Patch(facecolor=NODE_NORMAL_COLOR, edgecolor="white", label="Normal node"),
        Patch(facecolor=NODE_ATTACK_COLOR, edgecolor="white", label="Attack node"),
        Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="white",
               markeredgecolor="black", markeredgewidth=1.8, label="Mismatch"),
    ]

    if draw_relation_edges:
        relation_legend = [
            Line2D([0], [0], color=RELATION_TYPE_COLORS.get(r, "#BDBDBD"), lw=2,
                   label=RELATION_TYPE_NAMES.get(r, f"Relation {r}"))
            for r in sorted(set(_get_edge_arrays_from_data(data)[1].tolist()))
            if r in RELATION_TYPE_NAMES
        ]
    else:
        relation_legend = [Line2D([0], [0], color="#BDBDBD", lw=2, label="Graph edge")]

    fig.legend(
        handles=node_legend + relation_legend,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=min(7, len(node_legend + relation_legend)),
        frameon=False,
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.98])

    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight", format=save_path.split(".")[-1])
        print(f"[*] Saved graph layout visualization to: {save_path}")

    plt.show()
    plt.rcdefaults()


In [ ]:
# ------------------------------------------------------------
# Figure C: graph layout visualization for two selected cases
# ------------------------------------------------------------
# This cell draws exactly two graph-layout examples:
#   (a) a correctly predicted attack/mixed graph
#   (b) an imperfect attack/mixed graph with some correct attack detections and at least one node-level error
#
# Each graph-layout figure contains two panels:
#   left  = ground-truth node status
#   right = predicted node status

GRAPH_NODE_SIZE = 280          # increase to 220 if nodes are still too small; decrease to 150 if overlap is strong
GRAPH_FIGSIZE = (14, 6.2)

# ------------------------------------------------------------
# Case 1: successful / correctly predicted mixed graph
# ------------------------------------------------------------
plot_graph_layout_node_classification(
    data=data_node_attack,
    result=node_cls_attack,
    title='(a) Correctly predicted attack/mixed graph',
    save_path='../figures/explainability/node_classification_graph_layout_correct_case.pdf',
    figsize=GRAPH_FIGSIZE,
    layout_seed=42,
    draw_relation_edges=True,
    max_edges_per_relation=350,
    show_labels_for_attack_and_error=True,
    node_size=GRAPH_NODE_SIZE,
)

# ------------------------------------------------------------
# Case 2: imperfect / partially misclassified mixed graph
# ------------------------------------------------------------
if node_cls_error is not None:
    plot_graph_layout_node_classification(
        data=data_node_error,
        result=node_cls_error,
        title='(b) Partially misclassified attack/mixed graph',
        save_path='../figures/explainability/node_classification_graph_layout_imperfect_case.pdf',
        figsize=GRAPH_FIGSIZE,
        layout_seed=42,
        draw_relation_edges=True,
        max_edges_per_relation=350,
        show_labels_for_attack_and_error=True,
        node_size=GRAPH_NODE_SIZE,
    )
else:
    print('Skip imperfect graph-layout plot because no imperfect graph was selected.')


## Notes for interpretation

- **Correct case** cho thấy node head có thể định vị đúng attack nodes trong một mixed graph/window.
- **Imperfect case** cho thấy failure pattern điển hình: mô hình vẫn bắt đúng một số attack nodes, nhưng có thể bỏ sót hoặc báo nhầm một vài node.
- Trong mỗi graph, hai panel dùng cùng layout, vì vậy có thể so sánh trực tiếp vị trí node giữa ground truth và prediction.
- Node xanh là normal, node đỏ là attack.
- Node có viền đen hoặc dấu mismatch là node bị dự đoán sai giữa trạng thái normal/attack.
- Hai hình này phù hợp để đặt sau bảng node-classification metrics như một qualitative visualization.
